In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import glob
import warnings
from scipy.interpolate import make_interp_spline
warnings.filterwarnings('ignore')

# Set Arial font for plots
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False

# Set plot style
sns.set_style("whitegrid")

print("Packages imported successfully")

In [ ]:
# 1. Load cost data
cost_file = 'island_cost_summary_0.csv'
cost_df = pd.read_csv(cost_file)

print(f"Cost data shape: {cost_df.shape}")
print(f"Cost data columns: {list(cost_df.columns)}")
print("\nFirst 5 rows of cost data:")
print(cost_df.head())

# Check key columns
print(f"\nLatitude range: {cost_df['lat'].min():.2f} to {cost_df['lat'].max():.2f}")
print(f"Longitude range: {cost_df['lon'].min():.2f} to {cost_df['lon'].max():.2f}")
print(f"Renewable cost range: ${cost_df['renewable_cost'].min():,.2f} to ${cost_df['renewable_cost'].max():,.2f}")
print(f"Storage cost range: ${cost_df['storage_cost'].min():,.2f} to ${cost_df['storage_cost'].max():,.2f}")

In [ ]:
# 2. Load heating demand data
demand_folder = '../demand_get/data/get1/'
demand_files = glob.glob(os.path.join(demand_folder, 'demand_*.csv'))

print(f"Found {len(demand_files)} demand data files")

# Store heating demand data for each island
heating_demand_data = []

for file in demand_files:
    # Extract coordinates from filename
    filename = os.path.basename(file)
    coords = filename.replace('demand_', '').replace('.csv', '')
    lat, lon = map(float, coords.split('_'))
    
    # Read demand data
    demand_df = pd.read_csv(file)
    
    # Calculate heating demand statistics
    total_heating = demand_df['heating_demand'].sum()
    avg_heating = demand_df['heating_demand'].mean()
    max_heating = demand_df['heating_demand'].max()
    
    heating_demand_data.append({
        'lat': lat,
        'lon': lon,
        'total_heating_demand': total_heating,
        'avg_heating_demand': avg_heating,
        'max_heating_demand': max_heating
    })

# Convert to DataFrame
heating_df = pd.DataFrame(heating_demand_data)

print(f"Heating demand data shape: {heating_df.shape}")
print("\nFirst 5 rows of heating demand data:")
print(heating_df.head())

print(f"\nHeating demand ranges:")
print(f"Total heating demand: {heating_df['total_heating_demand'].min():.2f} to {heating_df['total_heating_demand'].max():.2f}")
print(f"Average heating demand: {heating_df['avg_heating_demand'].min():.2f} to {heating_df['avg_heating_demand'].max():.2f}")

In [ ]:
# 3. 直接根据经纬度合并数据
merged_df = pd.merge(cost_df, heating_df, on=['lat', 'lon'])

print(f"合并后数据形状: {merged_df.shape}")
print(f"匹配成功的岛屿数量: {len(merged_df)}")

print("\n合并数据前5行:")
print(merged_df[['lat', 'lon', 'renewable_cost', 'renewable_cost_per_capita', 'total_heating_demand', 'avg_heating_demand']].head())

In [ ]:
# 4. Clean style figure: Total heating demand vs energy costs
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from matplotlib.ticker import ScalarFormatter, FuncFormatter
import matplotlib.ticker as ticker

# Set Arial font and other parameters
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.minor.width'] = 0.3
plt.rcParams['ytick.minor.width'] = 0.3
plt.rcParams['axes.grid'] = False  # Disable grid by default

# --- NEW HELPER FUNCTION ---
def get_significance_stars(p_value):
    """Returns significance stars based on p-value."""
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return '' # Not significant

# Function to format axes with scientific notation
def set_axis_scientific(ax):
      ax.ticklabel_format(style='sci', axis='both', scilimits=(0, 0), useMathText=True)
      ax.xaxis.offsetText.set_fontsize(7)
      ax.xaxis.offsetText.set_fontfamily('Arial')
      ax.yaxis.offsetText.set_fontsize(7)
      ax.yaxis.offsetText.set_fontfamily('Arial')

# --- PLOTTING LOGIC ---

# Create figure and subplots
fig = plt.figure(figsize=(7.2, 5.4), dpi=300)
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.35)
axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), 
        fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]

# Calculate total energy system cost including LNG
merged_df['total_energy_cost_per_capita'] = (merged_df['renewable_cost_per_capita'] + 
                                             merged_df['storage_cost_per_capita'] + 
                                             merged_df['lng_cost_per_capita'])

# Define colors and data for each plot
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
plot_data = [
    ('renewable_cost_per_capita', 'Renewable Generation Cost\n($/per)'),
    ('storage_cost_per_capita', 'Energy Storage System Cost\n($/per)'),
    ('lng_cost_per_capita', 'LNG Cost\n($/per)'),
    ('total_energy_cost_per_capita', 'Total Cost\n($/per)')
]

# Store statistical results
stats_results = []

# Loop through each subplot to create the plots
for i, ax in enumerate(axes):
    cost_col, ylabel = plot_data[i]
    color = colors[i]
    x_data = merged_df['total_heating_demand']
    y_data = merged_df[cost_col]

    # Scatter plot
    ax.scatter(x_data, y_data, s=12, c=color, alpha=0.7, edgecolors='none')

    # Linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_data, y_data)
    
    # Store results for printing later
    stats_results.append({
        'name': ylabel.split('\n')[0],
        'r_squared': r_value**2,
        'p_value': p_value,
        'slope': slope,
        'intercept': intercept
    })

    # Regression line
    x_reg = np.linspace(x_data.min(), x_data.max(), 100)
    y_reg = slope * x_reg + intercept
    ax.plot(x_reg, y_reg, color='black', linewidth=1, alpha=0.8)

    # --- MODIFIED PART: Add R² with significance stars ---
    stars = get_significance_stars(p_value)
    ax.text(0.05, 0.95, f'R² = {r_value**2:.3f}{stars}', transform=ax.transAxes, 
            fontsize=7, verticalalignment='top', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

    # Set labels and ticks
    ax.set_xlabel('Total heating demand', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(axis='both', which='major', labelsize=7, width=0.5, length=3)

    # Style spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(1)
        ax.spines[spine].set_color("black")
    ax.grid(False)

    # Apply scientific notation
    set_axis_scientific(ax)

plt.show()

# --- PRINTING RESULTS ---
print("Statistical Analysis Results")
print("="*50)
for res in stats_results:
    stars = get_significance_stars(res['p_value'])
    print(f"{res['name']} regression: R² = {res['r_squared']:.3f}{stars}, p = {res['p_value']:.2e}")

print(f"\nRegression equations:")
for res in stats_results:
    print(f"{res['name']}: y = {res['slope']:.2e}x + {res['intercept']:.2f}")

print(f"\nSample size: n = {len(merged_df)} islands")
print(f"Heating demand range: {merged_df['total_heating_demand'].min():.0f} - {merged_df['total_heating_demand'].max():.0f}")
print(f"Total energy cost range: ${merged_df['total_energy_cost_per_capita'].min():.0f} - ${merged_df['total_energy_cost_per_capita'].max():.0f} per capita")


In [ ]:
# STL相关代码保留备用
def calculate_seasonal_variability_stl(merged_df):
    """使用STL分解计算每个岛屿的月度季节性能源输出变异性（备用方案）"""
    seasonal_data = []
    
    data_path = '../result/output_0'
    processed_count = 0
    failed_count = 0
    stl_success_count = 0
    fallback_count = 0
    
    print(f"开始使用STL分解处理 {len(merged_df)} 个岛屿的月度季节性分析...")
    
    # 添加tqdm进度条
    for _, island in tqdm(merged_df.iterrows(), total=len(merged_df), 
                         desc="Processing islands (STL)", unit="island", 
                         bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
        
        lat, lon = island['lat'], island['lon']
        
        # 构建输出文件路径
        output_file = f"{lat}_{lon}_results.csv"
        output_file_path = os.path.join(data_path, output_file)
        
        if os.path.exists(output_file_path):
            try:
                # 读取输出数据
                system_output = pd.read_csv(output_file_path)
                
                # 假设数据包含PV, WT, WEC列
                resources = ['PV', 'WT', 'WEC']
                seasonal_variations = {}
                resource_success = {}
                
                for resource in resources:
                    if resource in system_output.columns:
                        resource_data = system_output[resource].values
                        
                        # 使用3小时分辨率，计算月度季节性变异
                        # 创建时间索引
                        start_date = '2020-01-01'
                        time_steps = len(resource_data)
                        
                        if time_steps >= 240:  # 确保至少有一个月的数据（30天×8点/天=240点）
                            # 创建3小时间隔的时间索引
                            hours = pd.date_range(start=start_date, periods=time_steps, freq='3h')
                            ts = pd.Series(resource_data, index=hours)
                            
                            # 进行STL分解 - 使用月度周期
                            # 月度周期计算：约30天 × 8个点/天 = 240个点
                            monthly_period = int(365.25 * 8 / 12)  # 更精确的月度周期，约243个点
                            
                            try:
                                # STL分解 - 优化参数用于月度分析
                                stl = STL(ts, seasonal=7, period=monthly_period, trend=None)
                                result = stl.fit()
                                
                                # 提取季节性成分和趋势成分
                                seasonal_component = result.seasonal
                                trend_component = result.trend
                                
                                # 计算月度季节性变异性指标
                                # 方法1：月度季节性强度（季节性成分标准差与趋势平均值的比值）
                                trend_mean = np.nanmean(trend_component)
                                if trend_mean > 0:
                                    monthly_seasonality = np.nanstd(seasonal_component) / trend_mean
                                else:
                                    monthly_seasonality = 0
                                
                                # 使用月度季节性强度作为主要指标
                                seasonal_variations[f'{resource}_monthly_variability'] = monthly_seasonality
                                resource_success[resource] = 'STL'
                                
                            except Exception as e:
                                # 如果STL分解失败，使用简单的月度变异系数
                                monthly_data = ts.resample('M').mean()
                                if len(monthly_data) >= 3 and monthly_data.mean() > 0:  # 至少需要3个月的数据
                                    monthly_cv = monthly_data.std() / monthly_data.mean()
                                    seasonal_variations[f'{resource}_monthly_variability'] = monthly_cv
                                    resource_success[resource] = 'Fallback'
                                else:
                                    seasonal_variations[f'{resource}_monthly_variability'] = 0
                                    resource_success[resource] = 'Failed'
                        else:
                            # 数据不足，设为0
                            seasonal_variations[f'{resource}_monthly_variability'] = 0
                            resource_success[resource] = 'Insufficient_data'
                
                # 计算总体月度变异性（所有资源的加权平均）
                all_variations = [v for k, v in seasonal_variations.items() if k.endswith('_monthly_variability')]
                if all_variations:
                    total_seasonal_variability = np.mean(all_variations)
                else:
                    total_seasonal_variability = 0
                
                seasonal_data.append({
                    'lat': lat,
                    'lon': lon,
                    'total_seasonal_variability': total_seasonal_variability
                })
                
                processed_count += 1
                
                # 统计成功的方法
                stl_successes = sum(1 for method in resource_success.values() if method == 'STL')
                fallback_successes = sum(1 for method in resource_success.values() if method == 'Fallback')
                
                if stl_successes > 0:
                    stl_success_count += 1
                elif fallback_successes > 0:
                    fallback_count += 1
                
            except Exception as e:
                failed_count += 1
                continue
        else:
            failed_count += 1
    
    # 打印详细统计信息
    print(f"\n=== STL处理统计 ===")
    print(f"成功处理: {processed_count} 个岛屿")
    print(f"STL分解成功: {stl_success_count} 个岛屿")
    print(f"使用备选方案: {fallback_count} 个岛屿") 
    print(f"处理失败: {failed_count} 个岛屿")
    print(f"处理成功率: {processed_count/len(merged_df)*100:.1f}%")
    
    return pd.DataFrame(seasonal_data)

In [ ]:
"""
为每种能源资源（PV、WT、WEC）分别分析季节性差异与成本的关系
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import glob
import warnings
from tqdm import tqdm
from matplotlib.ticker import ScalarFormatter

warnings.filterwarnings('ignore')

# Set  Arial font for plots
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.minor.width'] = 0.3
plt.rcParams['ytick.minor.width'] = 0.3
plt.rcParams['axes.grid'] = False

def load_data():
    """加载成本和供暖需求数据"""
    # 1. Load cost data
    cost_file = 'island_cost_summary_0.csv'
    cost_df = pd.read_csv(cost_file)

    # 2. Load heating demand data
    demand_folder = '../demand_get/data/get1/'
    demand_files = glob.glob(os.path.join(demand_folder, 'demand_*.csv'))

    heating_demand_data = []
    for file in demand_files:
        filename = os.path.basename(file)
        coords = filename.replace('demand_', '').replace('.csv', '')
        lat, lon = map(float, coords.split('_'))
        
        demand_df = pd.read_csv(file)
        total_heating = demand_df['heating_demand'].sum()
        avg_heating = demand_df['heating_demand'].mean()
        max_heating = demand_df['heating_demand'].max()
        
        heating_demand_data.append({
            'lat': lat,
            'lon': lon,
            'total_heating_demand': total_heating,
            'avg_heating_demand': avg_heating,
            'max_heating_demand': max_heating
        })

    heating_df = pd.DataFrame(heating_demand_data)
    
    # 3. Merge data
    merged_df = pd.merge(cost_df, heating_df, on=['lat', 'lon'])
    
    # Calculate total energy system cost
    merged_df['total_energy_cost_per_capita'] = (merged_df['renewable_cost_per_capita'] + 
                                                 merged_df['storage_cost_per_capita'] + 
                                                 merged_df['lng_cost_per_capita'])
    
    return merged_df

def calculate_individual_seasonal_variability(merged_df):
    """为每种能源资源分别计算季节性差异 (使用变异系数CV方法)"""
    seasonal_data = []
    
    data_path = '../result/output_0'
    processed_count = 0
    failed_count = 0
    
    print(f"开始为每种能源资源单独处理 {len(merged_df)} 个岛屿的季节性分析...")
    
    for _, island in tqdm(merged_df.iterrows(), total=len(merged_df), 
                          desc="Processing islands", unit="island"):
        
        lat, lon = island['lat'], island['lon']
        
        output_file = f"{lat}_{lon}_results.csv"
        output_file_path = os.path.join(data_path, output_file)
        
        if os.path.exists(output_file_path):
            try:
                system_output = pd.read_csv(output_file_path)
                
                resources = ['PV', 'WT', 'WEC']
                resource_variability = {}
                
                for resource in resources:
                    seasonal_cv = 0.0 # Default to 0 (no variation)
                    if resource in system_output.columns:
                        resource_data = system_output[resource].values
                        
                        if len(resource_data) >= 240: # At least one month of data
                            start_date = '2020-01-01'
                            hours = pd.date_range(start=start_date, periods=len(resource_data), freq='3h')
                            ts = pd.Series(resource_data, index=hours)
                            
                            monthly_means = ts.resample('M').mean()
                            
                            # --- MODIFICATION START ---
                            # Filter out months with very low generation
                            monthly_mean_filtered = monthly_means[monthly_means > 0.01]

                            # Calculate Coefficient of Variation (CV)
                            if len(monthly_mean_filtered) >= 2:
                                seasonal_cv = monthly_mean_filtered.std() / monthly_mean_filtered.mean()
                            # --- MODIFICATION END ---

                    # Use the new column name convention '_seasonal_cv'
                    resource_variability[f'{resource}_seasonal_cv'] = seasonal_cv

                island_data = {'lat': lat, 'lon': lon}
                island_data.update(resource_variability)
                
                seasonal_data.append(island_data)
                processed_count += 1
                
            except Exception as e:
                print(f"处理岛屿 ({lat}, {lon}) 时发生错误: {e}")
                failed_count += 1
                continue
        else:
            failed_count += 1
    
    print(f"\n=== 处理统计 ===")
    print(f"成功处理: {processed_count} 个岛屿")
    print(f"处理失败: {failed_count} 个岛屿")
    print(f"处理成功率: {processed_count/len(merged_df)*100:.1f}%")
    
    return pd.DataFrame(seasonal_data)

def set_axis_scientific(ax):
      ax.ticklabel_format(style='sci', axis='both', scilimits=(0, 0), useMathText=True)
      ax.xaxis.offsetText.set_fontsize(7)
      ax.xaxis.offsetText.set_fontfamily('Arial')
      ax.yaxis.offsetText.set_fontsize(7)
      ax.yaxis.offsetText.set_fontfamily('Arial')

# --- 新增的辅助函数 ---
def get_significance_stars(p_value):
    """根据p值返回显著性星号"""
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''  # 不显著，返回空字符串

def plot_individual_resource_analysis(merged_seasonal_df, resource_name):
    """为单个能源资源生成四个成本对比图"""
    
    # 定义颜色
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
    
    # 创建图形
    fig = plt.figure(figsize=(7.2, 5.4), dpi=300)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.35)
    
    # 获取该资源的季节性差异数据
    x_data = merged_seasonal_df[f'{resource_name}_seasonal_cv']
    
    # 定义四种成本数据
    cost_data = [
        ('renewable_cost_per_capita', 'Renewable Generation Cost\n($/per)', colors[0]),
        ('storage_cost_per_capita', 'Energy Storage System Cost\n($/per)', colors[1]),
        ('lng_cost_per_capita', 'LNG Cost\n($/per)', colors[2]),
        ('total_energy_cost_per_capita', 'Total Cost\n($/per)', colors[3])
    ]
    
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
            fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
    
    statistics = []
    
    for i, (cost_col, ylabel, color) in enumerate(cost_data):
        ax = axes[i]
        y_data = merged_seasonal_df[cost_col]
        
        # 散点图
        ax.scatter(x_data, y_data, s=12, c=color, alpha=0.7, edgecolors='none')
        
        # 回归分析
        slope, intercept, r_value, p_value, std_err = stats.linregress(x_data, y_data)
        
        # 回归线
        x_reg = np.linspace(x_data.min(), x_data.max(), 100)
        y_reg = slope * x_reg + intercept
        ax.plot(x_reg, y_reg, color='black', linewidth=1, alpha=0.8)
        
        # 设置标签和格式
        ax.set_xlabel(f'{resource_name} seasonal variability', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7, width=0.5, length=3)
        
        # 移除顶部和右侧边框
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.spines['bottom'].set_linewidth(0.5)
        ax.grid(False)
        
        # --- 修改部分 ---
        # 获取显著性星号
        stars = get_significance_stars(p_value)
        
        # 添加R²和显著性星号的注释
        ax.text(0.05, 0.95, f'R² = {r_value**2:.3f}{stars}', transform=ax.transAxes, 
                fontsize=7, verticalalignment='top', 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))
        # --- 修改结束 ---
        
        # 设置科学计数法
        set_axis_scientific(ax)
        
        # 设置边框
        for spine in ['left', 'bottom']:
            ax.spines[spine].set_linewidth(1)
            ax.spines[spine].set_color("black")
        
        # 保存统计数据
        statistics.append({
            'cost_type': cost_col.replace('_per_capita', '').replace('_', ' ').title(),
            'r_squared': r_value**2,
            'p_value': p_value,
            'slope': slope,
            'intercept': intercept
        })
    
    plt.tight_layout()
    plt.show()
    
    return statistics

def main():
    """主函数"""
    print("正在加载数据...")
    merged_df = load_data()
    
    print("正在计算各能源资源的季节性差异...")
    seasonal_df = calculate_individual_seasonal_variability(merged_df)
    
    if len(seasonal_df) > 0:
        # 合并数据
        merged_seasonal_df = pd.merge(merged_df, seasonal_df, on=['lat', 'lon'])
        
        print(f"\n=== 数据合并结果 ===")
        print(f"总岛屿数: {len(merged_seasonal_df)}")
        
        # 为每种能源资源生成分析图
        resources = ['PV', 'WT', 'WEC']
        all_statistics = {}
        
        for resource in resources:
            if f'{resource}_seasonal_cv' in merged_seasonal_df.columns:
                print(f"\n正在生成 {resource} 的季节性差异分析图...")
                
                # 检查数据有效性
                valid_data = merged_seasonal_df[merged_seasonal_df[f'{resource}_seasonal_cv'] > 0.1]
                
                if len(valid_data) > 10:  # 至少需要10个有效数据点
                    stats_results = plot_individual_resource_analysis(valid_data, resource)
                    all_statistics[resource] = stats_results
                    
                    print(f"{resource} 分析完成:")
                    print(f"   有效数据点: {len(valid_data)} 个岛屿")
                    print(f"   季节性差异范围: {valid_data[f'{resource}_seasonal_cv'].min():.2f} - {valid_data[f'{resource}_seasonal_cv'].max():.2f}")
                    
                    for stat in stats_results:
                        # 在命令行输出也加上星号
                        stars = get_significance_stars(stat['p_value'])
                        print(f"   {stat['cost_type']}: R² = {stat['r_squared']:.3f}{stars}, p = {stat['p_value']:.2e}")
                else:
                    print(f"{resource} 的有效数据不足，跳过分析")
            else:
                print(f"缺少 {resource} 的季节性数据")
        
        # 打印总结
        print("\n=== 分析总结 ===")
        for resource, stats in all_statistics.items():
            print(f"\n{resource} 季节性差异与成本关系:")
            for stat in stats:
                stars = get_significance_stars(stat['p_value'])
                print(f"   {stat['cost_type']}: R² = {stat['r_squared']:.3f}{stars}")
    
    else:
        print("无法处理季节性数据")

main()

In [ ]:
"""
基于密度图的全球岛屿供暖需求和WT/WEC季节性差异可视化
Density-based Visualization of Global Island Heating Demand and WT/WEC Seasonal Variability
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm
import os
import glob
import warnings
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from scipy.ndimage import gaussian_filter

warnings.filterwarnings('ignore')

def load_and_merge_data():
    """加载并合并所有数据"""
    print("正在加载数据...")
    
    # 1. 加载成本数据
    cost_file = 'island_cost_summary_0.csv'
    cost_df = pd.read_csv(cost_file)
    
    # 2. 加载供暖需求数据
    demand_folder = '../demand_get/data/get1/'
    demand_files = glob.glob(os.path.join(demand_folder, 'demand_*.csv'))
    
    heating_demand_data = []
    for file in demand_files:  
        filename = os.path.basename(file)
        coords = filename.replace('demand_', '').replace('.csv', '')
        lat, lon = map(float, coords.split('_'))
        
        demand_df = pd.read_csv(file)
        total_heating = demand_df['heating_demand'].sum()
        
        heating_demand_data.append({
            'lat': lat,
            'lon': lon,
            'total_heating_demand': total_heating
        })
    
    heating_df = pd.DataFrame(heating_demand_data)
    
    # 3. 合并数据
    merged_df = pd.merge(cost_df, heating_df, on=['lat', 'lon'])
    
    # 4. 计算WT和WEC季节性差异 - 使用变异系数方法 (Coefficient of Variation)
    seasonal_data = []
    data_path = '../result/output_0'

    print(f"正在处理 {len(merged_df)} 个岛屿的WT和WEC数据...")

    for _, island in tqdm(merged_df.iterrows(), total=len(merged_df)):
        lat, lon = island['lat'], island['lon']
        output_file = f"{lat}_{lon}_results.csv"
        output_file_path = os.path.join(data_path, output_file)

        wt_seasonal_cv = 0.0  # 初始化WT变异系数
        wec_seasonal_cv = 0.0  # 初始化WEC变异系数

        if os.path.exists(output_file_path):
            try:
                system_output = pd.read_csv(output_file_path)

                # --- 计算 WT 季节性变异系数 ---
                if 'WT' in system_output.columns:
                    wt_data = system_output['WT'].values
                    if len(wt_data) >= 240:  # 至少需要240个数据点(约1个月)
                        start_date = '2020-01-01'
                        hours = pd.date_range(start=start_date, periods=len(wt_data), freq='3H')  # 3小时频率
                        ts = pd.Series(wt_data, index=hours)

                        # 按月分组计算平均值
                        monthly_means = ts.resample('M').mean()
                        # 过滤掉发电量很小的月份
                        monthly_mean_filtered = monthly_means[monthly_means > 0.01]

                        if len(monthly_mean_filtered) >= 2:  # 至少需要2个有效月份
                            # 计算变异系数 (标准差/平均值)
                            wt_seasonal_cv = monthly_mean_filtered.std() / monthly_mean_filtered.mean()
                        else:
                            wt_seasonal_cv = 0

                # --- 计算 WEC 季节性变异系数 ---
                if 'WEC' in system_output.columns:
                    wec_data = system_output['WEC'].values
                    if len(wec_data) >= 240:  # 至少需要240个数据点(约1个月)
                        start_date = '2020-01-01'
                        hours = pd.date_range(start=start_date, periods=len(wec_data), freq='3H')  # 3小时频率
                        ts = pd.Series(wec_data, index=hours)

                        # 按月分组计算平均值
                        monthly_means = ts.resample('M').mean()
                        # 过滤掉发电量很小的月份
                        monthly_mean_filtered = monthly_means[monthly_means > 0.01]

                        if len(monthly_mean_filtered) >= 2:  # 至少需要2个有效月份
                            # 计算变异系数 (标准差/平均值)
                            wec_seasonal_cv = monthly_mean_filtered.std() / monthly_mean_filtered.mean()
                        else:
                            wec_seasonal_cv = 0

            except:
                pass

        seasonal_data.append({
            'lat': lat,
            'lon': lon,
            'wt_seasonal_cv': wt_seasonal_cv,     # WT季节性变异系数
            'wec_seasonal_cv': wec_seasonal_cv   # WEC季节性变异系数
        })

    seasonal_df = pd.DataFrame(seasonal_data)
    final_df = pd.merge(merged_df, seasonal_df, on=['lat', 'lon'])

    print(f"最终数据: {len(final_df)} 个岛屿")
    return final_df

def visualize_heating_demand(data, ipcc_regions):
    """可视化全球岛屿供暖需求密度分布"""
    plt.rcParams['font.family'] = 'Arial'

    # 筛选有供暖需求的数据
    heating_data = data[data['total_heating_demand'] > 0].copy()

    if len(heating_data) == 0:
        print("没有供暖需求数据")
        return None

    print(f"可视化 {len(heating_data)} 个有供暖需求的岛屿")

    # 创建Point几何图形
    heating_data['geometry'] = heating_data.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
    geo_df = gpd.GeoDataFrame(heating_data, geometry='geometry')
    geo_df.set_crs(epsg=4326, inplace=True)

    # 对供暖需求使用对数变换以更好地显示差异
    geo_df['log_heating'] = np.log10(geo_df['total_heating_demand'] + 1)

    # 创建带有cartopy投影的图形
    fig = plt.figure(figsize=(14, 6), dpi=500)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    # 设置背景和地图特征
    ax.set_facecolor('#CCCCCC')
    ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.4)
    ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)
    
    if ipcc_regions is not None and not ipcc_regions.empty:
        # 遍历每个IPCC区域
        for _, region_row in ipcc_regions.iterrows():
            # (A) 绘制区域轮廓线
            ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                              facecolor='none',      # 设置为'none'以只显示边框
                              edgecolor='#555555',     # 边框颜色
                              linewidth=0.8,         # 边框线宽
                              linestyle='--',         # 虚线样式
                              zorder=3)              # zorder确保它在密度图之上

    # 创建密度图
    resolution = 1.0
    lons = np.arange(-180, 181, resolution)
    lats = np.arange(-90, 91, resolution)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    density_grid = np.zeros_like(lon_grid)
    all_lons = geo_df.geometry.x.values
    all_lats = geo_df.geometry.y.values
    all_values = geo_df['log_heating'].values
    bandwidth = 2.0
    for i in range(len(all_lons)):
        lon, lat = all_lons[i], all_lats[i]
        value = all_values[i]
        for x in range(len(lats)):
            for y in range(len(lons)):
                grid_lon = lons[y]
                grid_lat = lats[x]
                dist = np.sqrt((grid_lon - lon)**2 + (grid_lat - lat)**2)
                if dist <= 3 * bandwidth:
                    weight = np.exp(-0.5 * (dist / bandwidth)**2) * value
                    density_grid[x, y] += weight
    density_grid = gaussian_filter(density_grid, sigma=1.0)
    threshold = np.max(density_grid) * 0.2
    mask = density_grid < threshold
    upper_limit = np.percentile(density_grid[~mask], 99)
    density_grid_capped = np.clip(density_grid, 0, upper_limit)
    density_grid_masked = np.ma.array(density_grid_capped, mask=mask)
    heatmap_colors = ['#fff5f0', '#fee0d2', '#fcbba1', '#fc9272', '#fb6a4a', '#ef3b2c', '#cb181d', '#99000d']
    heatmap_cmap = mpl.colors.LinearSegmentedColormap.from_list('heating_palette', heatmap_colors)
    heatmap_norm = mpl.colors.Normalize(vmin=threshold, vmax=upper_limit)
    contour = ax.contourf(
        lon_grid, lat_grid, density_grid_masked,
        transform=ccrs.PlateCarree(), cmap=heatmap_cmap,
        norm=heatmap_norm, levels=8, alpha=0.7, zorder=2, extend='neither'
    )
    
    # 设置颜色映射 - 红色系
    sci_colors = ['#fff5f0', '#fcbba1', '#fc9272', '#ef3b2c', '#99000d']
    custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('heating_sci_palette', sci_colors)

    # 设置数值范围 - 使用99分位数避免极值影响
    min_heating = geo_df['total_heating_demand'].min()
    max_heating = np.percentile(geo_df['total_heating_demand'], 99)
    norm = mpl.colors.LogNorm(vmin=max(1, min_heating), vmax=max_heating)

    min_size = 20
    max_size = 200

    geo_df = geo_df.sort_values(by='total_heating_demand')

    # 创建一个假的散点图对象，仅用于生成颜色条
    fake_scatter = ax.scatter(
        [-1000], [-1000], c=[min_heating],
        cmap=custom_cmap, norm=norm, s=1
    )

    # 绘制带边框的散点图
    for idx, row in geo_df.iterrows():
        lon, lat = row.geometry.x, row.geometry.y
        value = row['total_heating_demand']
        normalized_value = min(value, max_heating)
        size = min_size + ((np.log10(normalized_value + 1) / np.log10(max_heating + 1)) * (max_size - min_size))
        color = custom_cmap(norm(value))
        temp_fig = plt.figure(figsize=(1, 1), frameon=False, dpi=300)
        temp_fig.patch.set_alpha(0)
        temp_ax = temp_fig.add_subplot(111)
        temp_ax.set_aspect('equal')
        temp_ax.patch.set_alpha(0)
        outer_circle = plt.Circle((0.5, 0.5), 0.18, color='black', alpha=1)
        temp_ax.add_patch(outer_circle)
        inner_circle = plt.Circle((0.5, 0.5), 0.15, color=color, alpha=1)
        temp_ax.add_patch(inner_circle)
        temp_ax.set_xlim(0, 1)
        temp_ax.set_ylim(0, 1)
        temp_ax.axis('off')
        temp_fig.tight_layout(pad=0)
        temp_fig.canvas.draw()
        point_img = np.array(temp_fig.canvas.renderer.buffer_rgba())
        plt.close(temp_fig)
        x, y = ax.projection.transform_point(lon, lat, src_crs=ccrs.PlateCarree())
        zoom_factor = np.sqrt(size) / 250
        imagebox = OffsetImage(point_img, zoom=zoom_factor)
        imagebox.image.axes = ax
        ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0, zorder=10)
        ax.add_artist(ab)

    # 添加颜色条
    cbar = fig.colorbar(fake_scatter, ax=ax, orientation='horizontal', shrink=0.6, pad=0.03, aspect=50)
    cbar.set_label('Heating Demand (kWh)', fontsize=24)

    # 手动设置更直观、更有意义的主刻度值
    tick_values = [100, 10000, 500000, 4000000] 
    cbar.set_ticks(tick_values)

    # 根据手动设置的刻度值，创建对应的标签
    tick_labels = ['100', '10K', '500K', '4M']
    cbar.set_ticklabels(tick_labels)

    # 关闭次要刻度线，消除杂乱感
    cbar.ax.minorticks_off()
    
    cbar.ax.tick_params(labelsize=18)

    ax.set_global()
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.axis('off')

    plt.tight_layout(pad=0)
    plt.show()

    return fig

def visualize_seasonal_variability(data, ipcc_regions, energy_type='wt'):
      """可视化全球岛屿季节性变化密度分布
      
      Parameters:
      data: DataFrame - 包含岛屿数据
      energy_type: str - 'wt' for wind turbine, 'wec' for wave energy converter
      """
      plt.rcParams['font.family'] = 'Arial'

      # 根据能源类型设置参数
      if energy_type == 'wt':
          ratio_col = 'wt_seasonal_cv'  # 修改为变异系数
          label = 'WT Seasonal Variability'
          # 蓝色系
          heatmap_colors = ['#f7fbff', '#deebf7', '#c6dbef', '#9ecae1', '#6baed6', '#4292c6',    '#2171b5', '#084594']
          sci_colors = ['#f7fbff', '#c6dbef', '#6baed6', '#2171b5', '#084594']
          palette_name = 'wt_palette'
      elif energy_type == 'wec':
          ratio_col = 'wec_seasonal_cv'  # 修改为变异系数
          label = 'WEC Seasonal Variability'
          # 绿色系
          heatmap_colors =['#f7fcf5','#e5f5e0','#c7e9c0','#a1d99b','#74c476','#41ab5d','#238b45','#005a32']
          sci_colors = ['#f7fcf5', '#c7e9c0', '#74c476', '#238b45', '#005a32']
          palette_name = 'wec_palette'
      else:
          raise ValueError("energy_type must be 'wt' or 'wec'")

      # 筛选有明显季节性变化的数据 (变异系数>0.1)
      seasonal_data = data[data[ratio_col] > 0.1].copy()

      if len(seasonal_data) == 0:
          print(f"没有明显的{energy_type.upper()}季节性数据")
          return None

      print(f"可视化 {len(seasonal_data)} 个有明显{energy_type.upper()}季节性变化的岛屿")        

      # 创建Point几何图形
      seasonal_data['geometry'] = seasonal_data.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
      geo_df = gpd.GeoDataFrame(seasonal_data, geometry='geometry')
      geo_df.set_crs(epsg=4326, inplace=True)

      # 创建带有cartopy投影的图形
      fig = plt.figure(figsize=(14, 6), dpi=500)
      ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

      # 设置背景和地图特征
      ax.set_facecolor('#CCCCCC')
      ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.4)
      ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
      ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
      ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)
      
      if ipcc_regions is not None and not ipcc_regions.empty:
        # 遍历每个IPCC区域
        for _, region_row in ipcc_regions.iterrows():
            # (A) 绘制区域轮廓线
            ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                              facecolor='none',      # 设置为'none'以只显示边框
                              edgecolor='#555555',     # 边框颜色
                              linewidth=0.8,         # 边框线宽
                              linestyle='--',         # 虚线样式
                              zorder=1)              # zorder确保它在密度图之上
                    
      # 创建密度图
      resolution = 1.0
      lons = np.arange(-180, 181, resolution)
      lats = np.arange(-90, 91, resolution)

      lon_grid, lat_grid = np.meshgrid(lons, lats)
      density_grid = np.zeros_like(lon_grid)

      all_lons = geo_df.geometry.x.values
      all_lats = geo_df.geometry.y.values
      all_values = geo_df[ratio_col].values

      bandwidth = 2.0

      # 计算每个网格点的密度值
      for i in range(len(all_lons)):
          lon, lat = all_lons[i], all_lats[i]
          value = all_values[i]

          for x in range(len(lats)):
              for y in range(len(lons)):
                  grid_lon = lons[y]
                  grid_lat = lats[x]

                  dist = np.sqrt((grid_lon - lon)**2 + (grid_lat - lat)**2)

                  if dist <= 3 * bandwidth:
                      weight = np.exp(-0.5 * (dist / bandwidth)**2) * value
                      density_grid[x, y] += weight

      density_grid = gaussian_filter(density_grid, sigma=1.0)

      threshold = np.max(density_grid) * 0.2
      mask = density_grid < threshold
      upper_limit = np.percentile(density_grid[~mask], 99)
      density_grid_capped = np.clip(density_grid, 0, upper_limit)
      density_grid_masked = np.ma.array(density_grid_capped, mask=mask)

      # 热力图颜色映射
      heatmap_cmap = mpl.colors.LinearSegmentedColormap.from_list(palette_name,heatmap_colors)
      heatmap_norm = mpl.colors.Normalize(vmin=threshold, vmax=upper_limit)

      # 绘制密度图
      contour = ax.contourf(
          lon_grid, lat_grid, density_grid_masked,
          transform=ccrs.PlateCarree(),
          cmap=heatmap_cmap,
          norm=heatmap_norm,
          levels=8,
          alpha=0.7,
          zorder=2,
          extend='neither'
      )

      # 设置点颜色映射
      custom_cmap =mpl.colors.LinearSegmentedColormap.from_list(f'{energy_type}_sci_palette', sci_colors)

      # 设置数值范围 - 使用99分位数避免极值影响
      min_val = geo_df[ratio_col].min()
      max_val = np.percentile(geo_df[ratio_col], 99)
      norm = mpl.colors.Normalize(vmin=min_val, vmax=max_val)

      min_size = 20
      max_size = 200

      geo_df = geo_df.sort_values(by=ratio_col)

      fake_scatter = ax.scatter(
          [-1000], [-1000],
          c=[min_val],
          cmap=custom_cmap,
          norm=norm,
          s=1
      )

      # 为每个点创建并添加标记
      for idx, row in geo_df.iterrows():
          lon, lat = row.geometry.x, row.geometry.y
          value = row[ratio_col]

          # 使用99分位数标准化点大小，避免极值影响
          normalized_value = min(value, max_val)  # 限制在99分位数以内
          size = min_size + (((normalized_value - min_val) / (max_val - min_val)) * (max_size - min_size))
          color = custom_cmap(norm(value))

          # 创建临时图形来生成点图像
          temp_fig = plt.figure(figsize=(1, 1), frameon=False, dpi=300)
          temp_fig.patch.set_alpha(0)

          temp_ax = temp_fig.add_subplot(111)
          temp_ax.set_aspect('equal')
          temp_ax.patch.set_alpha(0)

          # 黑色边框
          outer_circle = plt.Circle((0.5, 0.5), 0.18, color='black', alpha=1)
          temp_ax.add_patch(outer_circle)

          # 内圆
          inner_circle = plt.Circle((0.5, 0.5), 0.15, color=color, alpha=1)
          temp_ax.add_patch(inner_circle)

          temp_ax.set_xlim(0, 1)
          temp_ax.set_ylim(0, 1)
          temp_ax.axis('off')

          temp_fig.tight_layout(pad=0)
          temp_fig.canvas.draw()
          point_img = np.array(temp_fig.canvas.renderer.buffer_rgba())
          plt.close(temp_fig)

          x, y = ax.projection.transform_point(lon, lat, src_crs=ccrs.PlateCarree())
          zoom_factor = np.sqrt(size) / 250

          imagebox = OffsetImage(point_img, zoom=zoom_factor)
          imagebox.image.axes = ax

          ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0, zorder=10)
          ax.add_artist(ab)

      # 添加颜色条 - 清理刻度线
      cbar = fig.colorbar(fake_scatter, ax=ax, orientation='horizontal', shrink=0.6, pad=0.03, aspect=50)
      cbar.set_label(label, fontsize=24)

      # 设置清晰的刻度线 - 只显示3-4个主要刻度
      tick_values = np.linspace(min_val, max_val, 8)
      cbar.set_ticks(tick_values)
      cbar.set_ticklabels([f'{val:.1f}' for val in tick_values])
      cbar.ax.tick_params(labelsize=18)

      ax.set_global()
      ax.set_xticks([])
      ax.set_yticks([])
      for spine in ax.spines.values():
          spine.set_visible(False)
      plt.axis('off')

      plt.tight_layout(pad=0)
      plt.show()

      return fig

def visualize_combined_wt_wec_without_density(data, ipcc_regions):
    """可视化WT和WEC季节性变化的组合图 - 不包含密度背景"""
    plt.rcParams['font.family'] = 'Arial'

    # 筛选有显著季节性变化的数据 (变异系数>0.1)
    wt_data = data[data['wt_seasonal_cv'] > 0.1].copy()
    wec_data = data[data['wec_seasonal_cv'] > 0.1].copy()

    print(f"可视化 {len(wt_data)} 个有WT季节性变化的岛屿")
    print(f"可视化 {len(wec_data)} 个有WEC季节性变化的岛屿")

    if len(wt_data) == 0 and len(wec_data) == 0:
        print("没有明显的季节性数据")
        return None

    # 创建Point几何图形
    if len(wt_data) > 0:
        wt_data['geometry'] = wt_data.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
        wt_geo_df = gpd.GeoDataFrame(wt_data, geometry='geometry')
        wt_geo_df.set_crs(epsg=4326, inplace=True)

    if len(wec_data) > 0:
        wec_data['geometry'] = wec_data.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
        wec_geo_df = gpd.GeoDataFrame(wec_data, geometry='geometry')
        wec_geo_df.set_crs(epsg=4326, inplace=True)

    # 创建带有cartopy投影的图形
    fig = plt.figure(figsize=(14, 6), dpi=500)  # 高分辨率设置
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    # 设置背景和地图特征
    ax.set_facecolor('#CCCCCC')  # 浅灰色背景
    ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.4)  # 陆地颜色
    ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)  # 海洋颜色
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)  # 海岸线宽度
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)  # 国界线样式

    # 添加IPCC区域边界（如果有的话）
    if ipcc_regions is not None and not ipcc_regions.empty:
        for _, region_row in ipcc_regions.iterrows():
            ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                              facecolor='none',      # 不填充区域
                              edgecolor='#555555',   # 边框颜色
                              linewidth=0.8,         # 边框线宽
                              linestyle='--',        # 虚线样式
                              zorder=1)              # 图层顺序

    # 设置WT颜色映射 - 蓝色系
    wt_colors = ['#c6dbef', '#6baed6', '#2171b5']
    wt_cmap = mpl.colors.LinearSegmentedColormap.from_list('wt_palette', wt_colors)

    # 设置WEC颜色映射 - 绿色系
    wec_colors = [ '#c7e9c0', '#74c476', '#238b45']
    wec_cmap = mpl.colors.LinearSegmentedColormap.from_list('wec_palette', wec_colors)

    min_size = 40   # 最小点大小
    max_size = 400  # 最大点大小

    # 绘制WEC数据点 - 使用三角形区分
    if len(wec_data) > 0:
        min_wec = wec_geo_df['wec_seasonal_cv'].min()
        max_wec = np.percentile(wec_geo_df['wec_seasonal_cv'], 99)  # 使用99分位数避免极值
        wec_norm = mpl.colors.Normalize(vmin=min_wec, vmax=max_wec)

        # 按数值排序，小值在后绘制
        wec_geo_df = wec_geo_df.sort_values(by='wec_seasonal_cv')

        for idx, row in wec_geo_df.iterrows():
            lon, lat = row.geometry.x, row.geometry.y
            value = row['wec_seasonal_cv']

            # 计算点大小
            normalized_value = min(value, max_wec)
            size = min_size + (((normalized_value - min_wec) / (max_wec - min_wec)) * (max_size - min_size))
            color = wec_cmap(wec_norm(value))

            # 创建带边框的三角形标记
            temp_fig = plt.figure(figsize=(1, 1), frameon=False, dpi=300)
            temp_fig.patch.set_alpha(0)  # 透明背景
            temp_ax = temp_fig.add_subplot(111)
            temp_ax.set_aspect('equal')
            temp_ax.patch.set_alpha(0)

            # 三角形顶点坐标
            triangle_outer = np.array([[0.5, 0.8], [0.2, 0.2], [0.8, 0.2]])
            triangle_inner = np.array([[0.5, 0.75], [0.25, 0.25], [0.75, 0.25]])

            # 外圈（黑色边框）
            triangle_patch_outer = plt.Polygon(triangle_outer, color='black', alpha=0.8)
            temp_ax.add_patch(triangle_patch_outer)
            # 内圈（数据颜色）
            triangle_patch_inner = plt.Polygon(triangle_inner, color=color, alpha=0.8)
            temp_ax.add_patch(triangle_patch_inner)

            temp_ax.set_xlim(0, 1)
            temp_ax.set_ylim(0, 1)
            temp_ax.axis('off')

            temp_fig.tight_layout(pad=0)
            temp_fig.canvas.draw()
            point_img = np.array(temp_fig.canvas.renderer.buffer_rgba())
            plt.close(temp_fig)

            # 投影坐标转换
            x, y = ax.projection.transform_point(lon, lat, src_crs=ccrs.PlateCarree())
            zoom_factor = np.sqrt(size) / 250  # 缩放因子

            imagebox = OffsetImage(point_img, zoom=zoom_factor)
            imagebox.image.axes = ax
            ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0, zorder=9)  # WEC在上层
            ax.add_artist(ab)
            
    # 绘制WT数据点
    if len(wt_data) > 0:
        min_wt = wt_geo_df['wt_seasonal_cv'].min()
        max_wt = np.percentile(wt_geo_df['wt_seasonal_cv'], 99)  # 使用99分位数避免极值
        wt_norm = mpl.colors.Normalize(vmin=min_wt, vmax=max_wt)

        # 按数值排序，小值在后绘制
        wt_geo_df = wt_geo_df.sort_values(by='wt_seasonal_cv')

        for idx, row in wt_geo_df.iterrows():
            lon, lat = row.geometry.x, row.geometry.y
            value = row['wt_seasonal_cv']

            # 计算点大小
            normalized_value = min(value, max_wt)
            size = min_size + (((normalized_value - min_wt) / (max_wt - min_wt)) * (max_size - min_size))
            color = wt_cmap(wt_norm(value))

            # 创建带边框的圆形标记
            temp_fig = plt.figure(figsize=(1, 1), frameon=False, dpi=300)
            temp_fig.patch.set_alpha(0)  # 透明背景
            temp_ax = temp_fig.add_subplot(111)
            temp_ax.set_aspect('equal')
            temp_ax.patch.set_alpha(0)

            # 外圈（黑色边框）
            outer_circle = plt.Circle((0.5, 0.5), 0.18, color='black', alpha=0.6)
            temp_ax.add_patch(outer_circle)
            # 内圈（数据颜色）
            inner_circle = plt.Circle((0.5, 0.5), 0.15, color=color, alpha=0.6)
            temp_ax.add_patch(inner_circle)

            temp_ax.set_xlim(0, 1)
            temp_ax.set_ylim(0, 1)
            temp_ax.axis('off')

            temp_fig.tight_layout(pad=0)
            temp_fig.canvas.draw()
            point_img = np.array(temp_fig.canvas.renderer.buffer_rgba())
            plt.close(temp_fig)

            # 投影坐标转换
            x, y = ax.projection.transform_point(lon, lat, src_crs=ccrs.PlateCarree())
            zoom_factor = np.sqrt(size) / 250  # 缩放因子

            imagebox = OffsetImage(point_img, zoom=zoom_factor)
            imagebox.image.axes = ax
            ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0, zorder=10)
            ax.add_artist(ab)


    # 设置全球视图
    ax.set_global()
    ax.set_xticks([])  # 移除x轴刻度
    ax.set_yticks([])  # 移除y轴刻度
    
    # 移除图框
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.axis('off')  # 关闭坐标轴

    plt.tight_layout(pad=0)  # 紧密布局
    plt.show()

    return fig

def create_combined_legend():
    """
    创建WT和WEC组合图的独立图例，
    并将对应的标记（圆形/三角形）放置在每个颜色条的左侧。
    """
    plt.rcParams['font.family'] = 'Arial'
    
    # 调整figsize以适应新的布局
    fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), dpi=300)
    
    # --- 1. WT图例 (顶部) ---
    ax_wt = axes[0]
    wt_colors = [ '#c6dbef','#6baed6', '#2171b5']
    wt_cmap = mpl.colors.LinearSegmentedColormap.from_list('wt_palette', wt_colors)

    # 首先，在画布(ax_wt)上绘制我们想要的标记
    # a. 定义画布的坐标系范围为0到1
    ax_wt.set_xlim(0, 1)
    ax_wt.set_ylim(0, 1)
    # b. 在左侧绘制一个带黑色边框的圆形标记
    ax_wt.scatter([0.1], [0.5], marker='o', s=350, c=wt_colors[-1], 
                  edgecolor='black', linewidth=1.5, zorder=3)
    
    # 接下来，让画布本身变得不可见（隐藏背景、边框和刻度）
    ax_wt.patch.set_visible(False)
    for spine in ax_wt.spines.values():
        spine.set_visible(False)
    ax_wt.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
    
    # 然后，在画布的指定位置创建一个新的"内嵌"axes用于颜色条
    # [左边距, 下边距, 宽度, 高度]，坐标是相对于画布(ax_wt)的比例
    cax_wt = ax_wt.inset_axes([0.25, 0.45, 0.7, 0.1]) 
    norm_wt = mpl.colors.Normalize(vmin=0, vmax=1)
    
    # 在这个新的内嵌axes上创建颜色条
    wt_cbar = mpl.colorbar.ColorbarBase(cax_wt, cmap=wt_cmap, norm=norm_wt, orientation='horizontal')
    
    # 设置颜色条的标签和刻度
    wt_cbar.set_label('WT Seasonal Variability', fontsize=20, fontfamily='Arial', labelpad=10)
    wt_cbar.ax.tick_params(labelsize=16)
    wt_ticks = np.linspace(0, 1, 6)
    wt_cbar.set_ticks(wt_ticks)
    wt_cbar.set_ticklabels([f'{val:.1f}' for val in wt_ticks])

    # --- 2. WEC图例 (底部)，重复相同的过程 ---
    ax_wec = axes[1]
    wec_colors = [ '#c7e9c0','#74c476', '#238b45',]
    wec_cmap = mpl.colors.LinearSegmentedColormap.from_list('wec_palette', wec_colors)
    
    # 在画布(ax_wec)上绘制三角形标记
    ax_wec.set_xlim(0, 1)
    ax_wec.set_ylim(0, 1)
    ax_wec.scatter([0.1], [0.5], marker='^', s=350, c=wec_colors[-1], 
                   edgecolor='black', linewidth=1.5, zorder=3)

    # 让画布本身变得不可见
    ax_wec.patch.set_visible(False)
    for spine in ax_wec.spines.values():
        spine.set_visible(False)
    ax_wec.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)

    # 在画布的指定位置创建内嵌axes用于颜色条
    cax_wec = ax_wec.inset_axes([0.25, 0.45, 0.7, 0.1])
    norm_wec = mpl.colors.Normalize(vmin=0, vmax=1)
    
    # 创建颜色条
    wec_cbar = mpl.colorbar.ColorbarBase(cax_wec, cmap=wec_cmap, norm=norm_wec, orientation='horizontal')
    
    # 设置颜色条的标签和刻度
    wec_cbar.set_label('WEC Seasonal Variability', fontsize=20, fontfamily='Arial', labelpad=10)
    wec_cbar.ax.tick_params(labelsize=16)
    wec_ticks = np.linspace(0, 1, 6)
    wec_cbar.set_ticks(wec_ticks)
    wec_cbar.set_ticklabels([f'{val:.1f}' for val in wec_ticks])

    # --- 3. 最终布局调整 ---
    fig.tight_layout(h_pad=1) # 调整两个图例之间的垂直间距
    plt.show()

    return fig

def main():
    """主函数"""
    print("开始密度图可视化...")
    
    # 加载数据
    df = load_and_merge_data()
    ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")
    
    # 创建供暖需求密度图
    print("\n1. 创建供暖需求密度图...")
    heating_fig = visualize_heating_demand(df ,ipcc_regions)
    
    # 创建WT WEC季节性密度图
    print("\n2. 创建WT WEC季节性密度图...")
    # wt_fig = visualize_seasonal_variability(df, ipcc_regions, energy_type='wt')   # 风力发电
    # wec_fig = visualize_seasonal_variability(df, ipcc_regions, energy_type='wec') # 波浪发电
    
    # 创建WT和WEC组合可视化图（无密度背景）
    print("\n3. 创建WT和WEC组合季节性可视化图（无密度背景）...")
    # combined_fig = visualize_combined_wt_wec_without_density(df, ipcc_regions)
    
    # 创建独立图例
    print("\n4. 创建独立图例...")
    legend_fig = create_combined_legend()
    
    print("\n可视化完成！")
    
main()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# 确保在脚本开头设置字体，以便所有图都应用此设置
plt.rcParams['font.family'] = 'Arial'

def create_wt_legend():
    """
    创建独立的 WT 图例，并输出为单个图像。
    """
    # 创建一个只包含一个子图的图窗，调整figsize以适应单个图例
    fig, ax_wt = plt.subplots(1, 1, figsize=(8, 2.5), dpi=300)
    
    # --- WT 图例定义 ---
    wt_colors = ['#c6dbef', '#6baed6', '#2171b5']
    wt_cmap = mpl.colors.LinearSegmentedColormap.from_list('wt_palette', wt_colors)

    # a. 定义画布的坐标系范围为0到1
    ax_wt.set_xlim(0, 1)
    ax_wt.set_ylim(0, 1)
    # b. 在左侧绘制一个带黑色边框的圆形标记
    # ax_wt.scatter([0.1], [0.5], marker='o', s=350, c=wt_colors[-1], 
    #               edgecolor='black', linewidth=1.5, zorder=3)
    
    # 让画布本身变得不可见（隐藏背景、边框和刻度）
    ax_wt.patch.set_visible(False)
    for spine in ax_wt.spines.values():
        spine.set_visible(False)
    ax_wt.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
    
    # 在画布的指定位置创建一个新的"内嵌"axes用于颜色条
    cax_wt = ax_wt.inset_axes([0.25, 0.45, 0.7, 0.1]) 
    norm_wt = mpl.colors.Normalize(vmin=0, vmax=1)
    
    # 创建颜色条
    wt_cbar = mpl.colorbar.ColorbarBase(cax_wt, cmap=wt_cmap, norm=norm_wt, orientation='horizontal')
    
    # 设置颜色条的标签和刻度
    wt_cbar.set_label('WT Seasonal Variability', fontsize=28, fontfamily='Arial', labelpad=10)
    wt_cbar.ax.tick_params(labelsize=28)
    wt_ticks = mpl.ticker.LinearLocator(numticks=6) # 自动选择6个刻度
    wt_cbar.set_ticks(wt_ticks)
    wt_cbar.set_ticklabels([f'{val:.1f}' for val in wt_ticks.tick_values(0, 1)])

    # 调整布局并显示第一个图
    fig.tight_layout()
    plt.show()

    return fig

def create_wec_legend():
    """
    创建独立的 WEC 图例，并输出为单个图像。
    """
    # 创建一个只包含一个子图的新图窗
    fig, ax_wec = plt.subplots(1, 1, figsize=(8, 2.5), dpi=300)
    
    # --- WEC 图例定义 ---
    wec_colors = ['#c7e9c0', '#74c476', '#238b45']
    wec_cmap = mpl.colors.LinearSegmentedColormap.from_list('wec_palette', wec_colors)
    
    # 在画布(ax_wec)上绘制三角形标记
    ax_wec.set_xlim(0, 1)
    ax_wec.set_ylim(0, 1)
    # ax_wec.scatter([0.1], [0.5], marker='^', s=350, c=wec_colors[-1], 
    #                edgecolor='black', linewidth=1.5, zorder=3)

    # 让画布本身变得不可见
    ax_wec.patch.set_visible(False)
    for spine in ax_wec.spines.values():
        spine.set_visible(False)
    ax_wec.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)

    # 在画布的指定位置创建内嵌axes用于颜色条
    cax_wec = ax_wec.inset_axes([0.25, 0.45, 0.7, 0.1])
    norm_wec = mpl.colors.Normalize(vmin=0, vmax=1)
    
    # 创建颜色条
    wec_cbar = mpl.colorbar.ColorbarBase(cax_wec, cmap=wec_cmap, norm=norm_wec, orientation='horizontal')
    
    # 设置颜色条的标签和刻度
    wec_cbar.set_label('WEC Seasonal Variability', fontsize=28, fontfamily='Arial', labelpad=10)
    wec_cbar.ax.tick_params(labelsize=28)
    wec_ticks = mpl.ticker.LinearLocator(numticks=6) # 自动选择6个刻度
    wec_cbar.set_ticks(wec_ticks)
    wec_cbar.set_ticklabels([f'{val:.1f}' for val in wec_ticks.tick_values(0, 1)])

    # 调整布局并显示第二个图
    fig.tight_layout()
    plt.show()

    return fig

# --- 分别调用函数来生成和显示两个独立的图 ---
print("正在生成 WT 图例...")
wt_legend_fig = create_wt_legend()

print("\n正在生成 WEC 图例...")
wec_legend_fig = create_wec_legend()

In [ ]:
"""
结合最后一个cell的可视化经纬向分布的代码，画出上面三个全球地图的经纬向分布图
使用地图中相同的颜色方案：
1. 供暖需求分布图 - 红色系 (#fff5f0 到 #99000d)
2. WT季节性分布图 - 蓝色系 (#f7fbff 到 #084594) 
3. WEC季节性分布图 - 绿色系 (#f7fcf5 到 #005a32)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter
from scipy.interpolate import make_interp_spline
import os
import glob
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')
def create_longitude_profile_colored(data, column, ax=None, colors=None, label=''):
    """创建经度方向的彩色曲线图 (修正版)"""
    plt.rcParams['font.family'] = 'Arial'
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 1.5), dpi=500)
    
    if colors is None:
        colors = ['#ff0000', '#cc0000']
        
    lon_bins = np.arange(-180, 181, 5)
    lon_centers = (lon_bins[:-1] + lon_bins[1:]) / 2
    lon_values = []
    
    for i in range(len(lon_bins)-1):
        mask = (data['lon'] >= lon_bins[i]) & (data['lon'] < lon_bins[i+1])
        # --- 核心修正：将 .sum() 修改为 .mean() ---
        lon_values.append(data.loc[mask, column].mean() if mask.any() else 0)
    
    if len(lon_centers) > 3:
        lon_smooth = np.linspace(min(lon_centers), max(lon_centers), 300)
        spline = make_interp_spline(lon_centers, lon_values, k=3)
        lon_values_smooth = np.maximum(spline(lon_smooth), 0)
    else:
        lon_smooth = lon_centers
        lon_values_smooth = lon_values
    
    ax.plot(lon_smooth, lon_values_smooth, color=colors[0], linewidth=2.5, label=label)
    ax.fill_between(lon_smooth, 0, lon_values_smooth, alpha=0.3, color=colors[1])
    
    ax.set_xlim(-180, 180)
    ax.set_ylim(0, 1) # Y轴范围会自动适应0-1的范围
    ax.tick_params(axis='y', labelsize=28)
    ax.tick_params(axis='x', labelsize=28)
    
    # --- 修正：移除或注释掉强制科学计数的代码 ---
    # ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    # ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    # 
    # offset_text = ax.yaxis.get_offset_text()
    # offset_text.set_fontsize(28)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_xlabel('')
    ax.set_xticklabels([])
    ax.tick_params(axis='x', which='both', length=0)
    
    return ax
def create_latitude_profile_colored(data, column, ax=None, colors=None, label=''):
    """创建纬度方向的彩色曲线图 (修正版)"""
    plt.rcParams['font.family'] = 'Arial'
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(2, 10), dpi=500)
    
    if colors is None:
        colors = ['#ff0000', '#cc0000']
        
    lat_bins = np.arange(-90, 91, 2)
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    lat_values = []
    
    for i in range(len(lat_bins)-1):
        mask = (data['lat'] >= lat_bins[i]) & (data['lat'] < lat_bins[i+1])
        # --- 核心修正：将 .sum() 修改为 .mean() ---
        lat_values.append(data.loc[mask, column].mean() if mask.any() else 0)
    
    if len(lat_centers) > 3:
        lat_smooth = np.linspace(min(lat_centers), max(lat_centers), 300)
        spline = make_interp_spline(lat_centers, lat_values, k=3)
        lat_values_smooth = np.maximum(spline(lat_smooth), 0)
    else:
        lat_smooth = lat_centers
        lat_values_smooth = lat_values
    
    ax.plot(lat_values_smooth, lat_smooth, color=colors[0], linewidth=2.5, label=label)
    ax.fill_betweenx(lat_smooth, 0, lat_values_smooth, alpha=0.5, color=colors[1])
    
    ax.set_ylim(-90, 90)
    ax.set_xlim(0, 1) # X轴范围会自动适应0-1的范围
    ax.tick_params(axis='x', labelsize=28)
    ax.tick_params(axis='y', labelsize=28)
    
    # --- 修正：移除或注释掉强制科学计数的代码 ---
    # ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    # ax.ticklabel_format(axis='x', style='sci', scilimits=(0,0))
    # 
    # offset_text = ax.xaxis.get_offset_text()
    # offset_text.set_fontsize(28)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_ylabel('')
    ax.set_yticklabels([])
    ax.tick_params(axis='y', which='both', length=0)
    
    return ax

def load_and_process_data():
    """加载并处理所有数据"""
    print("正在加载数据...")
    
    # 1. 加载成本数据
    cost_file = 'island_cost_summary_0.csv'
    cost_df = pd.read_csv(cost_file)
    
    # 2. 加载供暖需求数据
    demand_folder = '../demand_get/data/get1/'
    demand_files = glob.glob(os.path.join(demand_folder, 'demand_*.csv'))
    
    heating_demand_data = []
    for file in demand_files:
        filename = os.path.basename(file)
        coords = filename.replace('demand_', '').replace('.csv', '')
        lat, lon = map(float, coords.split('_'))
        
        demand_df = pd.read_csv(file)
        total_heating = demand_df['heating_demand'].sum()
        
        heating_demand_data.append({
            'lat': lat,
            'lon': lon,
            'total_heating_demand': total_heating
        })
    
    heating_df = pd.DataFrame(heating_demand_data)
    
    # 3. 合并数据
    merged_df = pd.merge(cost_df, heating_df, on=['lat', 'lon'])
    
    # 4. 计算WT和WEC季节性差异
    seasonal_data = []
    data_path = '../result/output_0'
    
    print(f"正在处理 {len(merged_df)} 个岛屿的WT和WEC数据...")
    
    for _, island in tqdm(merged_df.iterrows(), total=len(merged_df), desc="处理季节性数据"):
        lat, lon = island['lat'], island['lon']
        output_file = f"{lat}_{lon}_results.csv"
        output_file_path = os.path.join(data_path, output_file)
        
        wt_seasonal_cv = 0.0  # 初始化WT变异系数
        wec_seasonal_cv = 0.0  # 初始化WEC变异系数
        
        if os.path.exists(output_file_path):
            try:
                system_output = pd.read_csv(output_file_path)
                
                # --- 计算 WT 季节性变异系数 ---
                if 'WT' in system_output.columns:
                    wt_data = system_output['WT'].values
                    if len(wt_data) >= 240:  # 至少需要240个数据点(约1个月)
                        start_date = '2020-01-01'
                        hours = pd.date_range(start=start_date, periods=len(wt_data), freq='3H')  # 3小时频率
                        ts = pd.Series(wt_data, index=hours)

                        # 按月分组计算平均值
                        monthly_means = ts.resample('M').mean()
                        # 过滤掉发电量很小的月份
                        monthly_mean_filtered = monthly_means[monthly_means > 0.01]

                        if len(monthly_mean_filtered) >= 2:  # 至少需要2个有效月份
                            # 计算变异系数 (标准差/平均值)
                            wt_seasonal_cv = monthly_mean_filtered.std() / monthly_mean_filtered.mean()
                        else:
                            wt_seasonal_cv = 0

                # --- 计算 WEC 季节性变异系数 ---
                if 'WEC' in system_output.columns:
                    wec_data = system_output['WEC'].values
                    if len(wec_data) >= 240:  # 至少需要240个数据点(约1个月)
                        start_date = '2020-01-01'
                        hours = pd.date_range(start=start_date, periods=len(wt_data), freq='3H')  # 3小时频率
                        ts = pd.Series(wec_data, index=hours)

                        # 按月分组计算平均值
                        monthly_means = ts.resample('M').mean()
                        # 过滤掉发电量很小的月份
                        monthly_mean_filtered = monthly_means[monthly_means > 0.01]

                        if len(monthly_mean_filtered) >= 2:  # 至少需要2个有效月份
                            # 计算变异系数 (标准差/平均值)
                            wec_seasonal_cv = monthly_mean_filtered.std() / monthly_mean_filtered.mean()
                        else:
                            wec_seasonal_cv = 0

            except:
                pass
        
        seasonal_data.append({
            'lat': lat,
            'lon': lon,
            'wt_seasonal_cv': wt_seasonal_cv,
            'wec_seasonal_cv': wec_seasonal_cv
        })
    
    seasonal_df = pd.DataFrame(seasonal_data)
    final_df = pd.merge(merged_df, seasonal_df, on=['lat', 'lon'])
    
    print(f"最终数据: {len(final_df)} 个岛屿")
    return final_df

def create_all_distribution_plots_combined():
    """
    创建供暖需求的独立分布图，以及WT和WEC组合的经纬向分布图。
    """
    plt.rcParams['font.family'] = 'Arial'
    
    data = load_and_process_data()
    
    # --- 1. 绘制独立的供暖需求分布图 ---
    print("\n--- 创建独立的供暖需求分布图 ---")
    heating_config = {
        'data': data[data['total_heating_demand'] > 0],
        'column': 'total_heating_demand',
        'title': 'Heating Demand Distribution',
        'colors': ['#cb181d', '#fc9272']
    }
    
    # 供暖经度图
    fig_lon_h, ax_lon_h = plt.subplots(figsize=(14, 1.5), dpi=300)
    create_longitude_profile_colored(heating_config['data'], heating_config['column'], ax_lon_h, heating_config['colors'])
    plt.show()
    
    # 供暖纬度图
    fig_lat_h, ax_lat_h = plt.subplots(figsize=(3, 10), dpi=300)
    create_latitude_profile_colored(heating_config['data'], heating_config['column'], ax_lat_h, heating_config['colors'])
    plt.show()

    # --- 2. 绘制组合的 WT 和 WEC 分布图 ---
    print("\n--- 创建组合的 WT 和 WEC 经纬度分布图 ---")
    
    wt_config = {
        'data': data[data['wt_seasonal_cv'] > 0.1],
        'column': 'wt_seasonal_cv',
        'colors': ['#2171b5', "#419dd2"]
    }
    wec_config = {
        'data': data[data['wec_seasonal_cv'] > 0.1],
        'column': 'wec_seasonal_cv',
        'colors': ['#238b45', "#89c68b"]
    }

    # --- 组合经度图 ---
    fig_lon_comb, ax_lon_comb = plt.subplots(figsize=(14, 2.5), dpi=300)
    
     # 在同一个ax上画WEC
    create_longitude_profile_colored(wec_config['data'], wec_config['column'], ax_lon_comb, wec_config['colors'], label='WEC')
    # 在同一个ax上画WT
    create_longitude_profile_colored(wt_config['data'], wt_config['column'], ax_lon_comb, wt_config['colors'], label='WT')
   
    
    # 添加图例和坐标轴标签
    # ax_lon_comb.legend(fontsize=24, frameon=False, loc='upper right')
    # ax_lon_comb.set_xticks([-180, -90, 0, 90, 180])
    # ax_lon_comb.set_xticklabels(['-180', '-90', '0', '90', '180'])
    # ax_lon_comb.tick_params(axis='x', which='both', length=5) # 恢复刻度线
    # ax_lon_comb.set_xlabel('Longitude', fontsize=28)
    
    plt.tight_layout()
    plt.show()

    # --- 组合纬度图 ---
    fig_lat_comb, ax_lat_comb = plt.subplots(figsize=(3, 10), dpi=300)
    
    # 在同一个ax上画WT
    create_latitude_profile_colored(wt_config['data'], wt_config['column'], ax_lat_comb, wt_config['colors'], label='WT')
    # 在同一个ax上画WEC
    create_latitude_profile_colored(wec_config['data'], wec_config['column'], ax_lat_comb, wec_config['colors'], label='WEC')

    # 添加图例和坐标轴标签
    # ax_lat_comb.legend(fontsize=24, frameon=False, loc='upper right')
    # ax_lat_comb.set_yticks([-90, -60, -30, 0, 30, 60, 90])
    # ax_lat_comb.set_yticklabels(['-90', '-60', '-30', '0', '30', '60', '90'])
    # ax_lat_comb.tick_params(axis='y', which='both', length=5) # 恢复刻度线
    # ax_lat_comb.set_ylabel('Latitude', fontsize=28)
    
    plt.tight_layout()
    plt.show()

# 执行创建经纬向分布图
create_all_distribution_plots_combined()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# 定义颜色列表
# 1. WT 季节性分布图 - 蓝色系
wt_colors = ['#c6dbef', '#6baed6', '#2171b5', '#084594']

# 2. WEC 季节性分布图 - 绿色系
wec_colors = [ '#c7e9c0', '#74c476', '#238b45', '#005a32']

# 创建颜色映射 (Colormap)
wt_cmap = mcolors.LinearSegmentedColormap.from_list('wt_palette', wt_colors)
wec_cmap = mcolors.LinearSegmentedColormap.from_list('wec_palette', wec_colors)

# 创建一个2行1列的图，用于展示两个颜色条
fig, axes = plt.subplots(2, 1, figsize=(8, 2), dpi=100)
fig.suptitle('Color Schemes', fontsize=16, y=1.05)

# 绘制第一个颜色条 (WT)
cbar_wt = fig.colorbar(plt.cm.ScalarMappable(cmap=wt_cmap), cax=axes[0], orientation='horizontal')
axes[0].set_title('WT Seasonal Variability (Blue Scheme)', loc='left')

# 绘制第二个颜色条 (WEC)
cbar_wec = fig.colorbar(plt.cm.ScalarMappable(cmap=wec_cmap), cax=axes[1], orientation='horizontal')
axes[1].set_title('WEC Seasonal Variability (Green Scheme)', loc='left')

# 调整布局并显示
plt.tight_layout()
plt.show()